# Lab: RAG on Rules of a new sport (Student Version)

Objective: build a simple and complete Retrieval-Augmented Generation (RAG) workflow using the Laws of the Game.

We will:
- Load a rules knowledge base
- Build a lightweight index (embeddings)
- Retrieve the most relevant rules for a question
- Produce a short answer from the retrieved context

This version has blanks for you to complete.

## Setup
We work with the files created in the exercise folder:
- rules.json (knowledge base)
- questions_mcq_student.json (questions)

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_DIR = Path('lab_rag_math/data/excercise')
RULES_PATH = BASE_DIR / 'rules.json'
QUESTIONS_PATH = BASE_DIR / 'questions_mcq_student.json'
CACHE_DIR = BASE_DIR / 'models_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)


First, let's ensure the `lab_rag_math/data/excercise` directory exists and download the `rules.json` and `questions_mcq_student.json` files.

In [ ]:
import os
import requests

# Ensure the base directory exists
BASE_DIR = Path('lab_rag_math/data/excercise')
os.makedirs(BASE_DIR, exist_ok=True)

# Download rules.json
rules_url = "https://raw.githubusercontent.com/ferrazzipietro/dataset-nlp-math/main/data_rag/excercise/rules.json"
rules_path = BASE_DIR / 'rules.json'
response = requests.get(rules_url)
response.raise_for_status() # Raise an exception for HTTP errors
with open(rules_path, 'wb') as f:
    f.write(response.content)
print(f"Downloaded rules.json to {rules_path}")

# Download questions_mcq_student.json
questions_url = "https://raw.githubusercontent.com/ferrazzipietro/dataset-nlp-math/main/data_rag/excercise/questions_mcq_student.json"
questions_path = BASE_DIR / 'questions_mcq_student.json'
response = requests.get(questions_url)
response.raise_for_status() # Raise an exception for HTTP errors
with open(questions_path, 'wb') as f:
    f.write(response.content)
print(f"Downloaded questions_mcq_student.json to {questions_path}")


Downloaded rules.json to lab_rag_math/data/excercise/rules.json
Downloaded questions_mcq_student.json to lab_rag_math/data/excercise/questions_mcq_student.json


## Build the Index
We embed each rule paragraph with a sentence embedding model.

In [ ]:

rules = json.loads(RULES_PATH.read_text(encoding='utf-8'))
questions = json.loads(QUESTIONS_PATH.read_text(encoding='utf-8'))

print('Rules:', len(rules))
print('Questions:', len(questions))

Rules: 49
Questions: 25


In [ ]:
# TODO: initialize the embedder
# Hint: use Qwen/Qwen3-Embedding-0.6B
embedder = None  # replace with a real model

# TODO: build the KB chunks with embeddings
KB_CHUNKS = []
for rule_id, text in rules.items():
    # TODO: compute embedding for text
    embedding = None  # replace
    KB_CHUNKS.append({
        "id": rule_id,
        "text": text,
        "embedding": embedding
    })

print('KB chunks:', len(KB_CHUNKS))

KB chunks: 49


## Retrieval
Implement the exact functions below (see the solution notebook for reference).

In [ ]:
# TODO: implement these functions
def embed_query(query):
    # return a single embedding for the query
    raise NotImplementedError()

def cosine_similarity_matrix(query, kb_emb):
    # return a 1 x N similarity matrix
    raise NotImplementedError()

def retrieve_top_n(query, kb_emb, n=3):
    # return top-n chunks with text and similarity
    raise NotImplementedError()

sample_q = questions[0]['question']
top = retrieve_top_n(sample_q, KB_CHUNKS, n=3)
print('Question:', sample_q)
for item in top:
    print(f"- similarity={item['similarity']:.3f}")
    print(f"  {item['text'][:160]}...")

## Generation: Answer from Retrieved Context
Complete the missing model loading and generation pieces.

In [ ]:
# TODO: load the generator model
GEMMA_ID = "unsloth/gemma-3-1B-it"
tokenizer_gemma = None  # replace
model_gemma = None  # replace

QUERIES = [{"query": q["question"]} for q in questions]

system_prompt = ""

RESULTS = []
for query in QUERIES:
    print('Query: ', query['query'])
    top_chunks = retrieve_top_n(query['query'], KB_CHUNKS, n=3)
    for retrieved in top_chunks:
        print(f"Similarity={retrieved['similarity']:.3f}, text={retrieved['text'][:180]}")
    context = [r['text'] for r in top_chunks]

    user_message = f"""Here is some useful context: {'\n'.join(context)}\n\n Here is the query: {query['query']}
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]

    # TODO: build model inputs from messages
    inputs = None  # replace

    # TODO: generate and decode the answer
    answer = None  # replace
    RESULTS.append({
        'query': query['query'],
        'retrieved': top_chunks,
        'answer': answer
    })
    print()

We do not have the labels to understand how the model performs. To have an idea about whether it works well, we are going to use an authomatic Judge (LLM-as-a-judge), to ask it:
1) whether the answer is grounded in the retrieved context
2) whether the retrieved context is reliable for the query.

By doing so, you can get an idea about the performances of the model, and you can use this results to do fix your RAG to get better results.

 Be careful, it can be the case that your Judge is wrong!

In [ ]:
def run_judge(prompt):
    # TODO: implement a function that runs the judge model on the prompt and returns the output
    raise NotImplementedError()

JUDGMENTS = []
for result in RESULTS:
    query = result["query"]
    answer = result["answer"]
    context = "\n".join([r["text"] for r in result["retrieved"]])

    grounding_prompt = f"""""" # TODO: replace with a prompt to evaluate if the answer is grounded in the context

    reliability_prompt = f"""""" # TODO: replace with a prompt to evaluate if the retrieved context is reliable for the query

    grounded = run_judge(grounding_prompt)
    reliable = run_judge(reliability_prompt)
    JUDGMENTS.append({
        "query": query,
        "grounding": grounded,
        "context_reliability": reliable
    })

print('Judged:', len(JUDGMENTS))
print('Sample:', JUDGMENTS[0] if JUDGMENTS else None)

overall_score = None # TODO: replace with a function of the judgments to compute an overall score for the system, to get an estimate of the system performance on the task

print('Overall score:', overall_score)

# How to submit

- Generate a `.csv` with two columns, `id` and `prediction`. The prediction must be the letter identifying the correct answer (e.g., A). (you can find an example and download it from the top right button at https://github.com/ferrazzipietro/dataset-nlp-math/blob/main/data_rag/submission_rag.zip)
- Go to the CodaBench portal https://www.codabench.org/competitions/16004/
- Select "Lab 5 - RAG" under the "My Submissions" page
- Download the `.py` with your code and the `exercise_predictions.csv` file with your predictions and save them in the same directory. Go to the directory and **select the 2 files, and zip them together at once.**
- Submit the .zip file containing both the submission `.csv` file and the `.py` with your code.
- You should achieve an accuracy of at least 0.75
- We will take into account the number of submissions you do in CodaBench. The fewer, the better.